# NER Dataset Creation — Political & Economic Entities

## Overview
Acest notebook construieste un dataset NER pentru entitati politice si economice in engleza.

**Pipeline complet:**
1. Instalare dependinte
2. Colectare corpus (CC-News + Wikipedia + SEC EDGAR)
3. Segmentare in propozitii
4. Weak supervision cu Snorkel (LFs regex + gazetteers + Wikidata)
5. Extractie entitati si conversie in format NER (IOB2 spans)
6. Adnotare LLM cu Claude API (~2000 exemple)
7. [Manual] Doccano pentru gold standard (500-1000 exemple)
8. Split train / dev / test

---
**Timp estimat de rulare:** ~45-90 minute (depinde de conexiune si API calls)


## PASUL 1 — Instalare dependinte

Ruleaza aceasta celula prima oara si **restarteza runtime-ul** dupa ce se termina instalarea (Runtime > Restart Runtime).
Nu e nevoie sa o rulezi din nou dupa restart.

In [ ]:
!pip install -q snorkel datasets spacy tqdm wikipedia-api anthropic sec-edgar-downloader nltk
!python -m spacy download en_core_web_sm -q
print("\nInstalare completa. Daca esti in Colab, mergi la Runtime > Restart Runtime, apoi continua de la celula urmatoare.")

## PASUL 2 — Importuri si configurare

Definim labelurile NER si parametrii globali ai notebook-ului.

In [ ]:
import re
import json
import time
import random
import requests
import warnings
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# ─── Labeluri NER ────────────────────────────────────────────────────────────
# Acestea sunt toate labelurile pe care modelul le va recunoaste.
# Poti adauga labeluri noi fara sa re-antrenezi (avantajul GLiNER).
NER_LABELS = [
    "POLITICIAN",          # Joe Biden, Jerome Powell
    "POLITICAL_PARTY",     # Republican Party, Labour Party
    "POLITICAL_ORG",       # NATO, European Commission, UN
    "FINANCIAL_ORG",       # Federal Reserve, IMF, Goldman Sachs
    "ECONOMIC_INDICATOR",  # GDP, CPI, inflation rate
    "POLICY",              # Quantitative Easing, Green New Deal
    "LEGISLATION",         # Dodd-Frank Act, Inflation Reduction Act
    "MARKET_EVENT",        # 2008 financial crisis, Black Monday
    "CURRENCY",            # USD, EUR, Bitcoin
    "TRADE_AGREEMENT",     # NAFTA, TPP, USMCA
    "GPE",                 # United States, Eurozone, EU
    # GPE = Geopolitical entity, pentru locatii care au relevanta politica/economica
]

# ─── Parametri colectare corpus ──────────────────────────────────────────────
# Modifica aceste valori daca vrei mai multe/putine date
MAX_CC_NEWS_ARTICLES   = 5000   # articole din CC-News
MAX_WIKI_ARTICLES      = 500    # articole Wikipedia
MAX_SEC_FILINGS        = 200    # filings SEC EDGAR
MAX_SENTENCES_PER_DOC  = 20     # propozitii maxime per document
MIN_SENTENCE_LEN       = 30     # caractere minime per propozitie
MAX_SENTENCE_LEN       = 300    # caractere maxime per propozitie

# ─── Directoare output ───────────────────────────────────────────────────────
Path("dataset/raw").mkdir(parents=True, exist_ok=True)
Path("dataset/annotated").mkdir(parents=True, exist_ok=True)
Path("dataset/splits").mkdir(parents=True, exist_ok=True)

print("Configurare completa.")
print(f"Labeluri definite: {NER_LABELS}")

## PASUL 3 — Colectare corpus

Colectam text brut din trei surse complementare:
- **CC-News**: stiri generale, buna acoperire politica si economica
- **Wikipedia**: articole enciclopedice, entitati bine definite
- **SEC EDGAR**: rapoarte financiare, zona economica

Fiecare sursa este salvata separat in `dataset/raw/`.

In [ ]:
# ─── Sursa 1: CC-News (HuggingFace) ──────────────────────────────────────────
# CC-News contine milioane de articole de stiri din Common Crawl.
# Filtram dupa keywords relevante domeniului nostru.

from datasets import load_dataset

KEYWORDS = [
    # Economic
    "GDP", "inflation", "interest rate", "Federal Reserve", "IMF",
    "World Bank", "stock market", "bond yield", "fiscal policy",
    "monetary policy", "recession", "unemployment", "CPI", "trade deficit",
    # Political
    "president", "senator", "prime minister", "parliament", "election",
    "congress", "legislation", "sanctions", "NATO", "treaty", "minister",
    "European Commission", "White House", "G7", "G20",
]

print("Se incarca CC-News (streaming)...")
cc_news_dataset = load_dataset("cc_news", split="train", streaming=True, trust_remote_code=True)
# streaming = True (datele sunt descarcate on-the-fly -> metoda devine asincrona)
# trust_remote_code = True (acorda permisiuni scriptului python din biblioteca sa descarce, dezarhiveze si formateze articolele de pe serverle sursa, 
# folosind un script python intern)

cc_news_texts = []
pbar = tqdm(total=MAX_CC_NEWS_ARTICLES, desc="CC-News") # biblioteca ce arata progresul (bara de progres) a articolelor adaugate in dateset in timp real. 

for article in cc_news_dataset:
    text = article.get("text", "").strip()  # strip = elimina spatiile de inceput si final din text; get("text") -> extrage din dictionarul rezultat din CC-News doar 
                                            # campul text cu continutul articolului 
    if len(text) < 200:
        continue
    if any(kw.lower() in text.lower() for kw in KEYWORDS):  # daca apar orice indicator predefinit in text 
        cc_news_texts.append(text)      # adauga textul in dataset
        pbar.update(1)                  # actualizeaza progress barul pt datasetul final
        if len(cc_news_texts) >= MAX_CC_NEWS_ARTICLES:
            break

pbar.close()
print(f"Colectate {len(cc_news_texts)} articole CC-News.")

with open("dataset/raw/cc_news.json", "w") as f:
    json.dump(cc_news_texts, f)
print("Salvat: dataset/raw/cc_news.json")

In [ ]:
# ─── Sursa 2: Wikipedia ───────────────────────────────────────────────────────
# Colectam articole Wikipedia pe teme politice si economice.
# Wikipedia are entitati bine definite si consistente — ideal pentru NER.

import wikipediaapi

# TITLURI de ARTICOLE REPREZENTATIVE pentru domeniul nostru
WIKI_TOPICS = [
    # Politicieni
    "Joe Biden", "Donald Trump", "Jerome Powell", "Christine Lagarde",
    "Janet Yellen", "Emmanuel Macron", "Olaf Scholz", "Rishi Sunak",
    "Angela Merkel", "Barack Obama", "Xi Jinping", "Vladimir Putin",
    # Organizatii politice
    "NATO", "European Union", "United Nations", "European Commission",
    "Republican Party (United States)", "Democratic Party (United States)",
    "G7", "G20", "BRICS",
    # Organizatii economice
    "International Monetary Fund", "World Bank", "Federal Reserve",
    "European Central Bank", "Bank of England", "Goldman Sachs",
    "JPMorgan Chase", "BlackRock",
    # Concepte economice
    "Quantitative easing", "Inflation", "Gross domestic product",
    "Monetary policy", "Fiscal policy", "Interest rate",
    "Stock market crash", "2008 financial crisis", "Stagflation",
    # Legislatie / Tratate
    "Dodd–Frank Wall Street Reform and Consumer Protection Act",
    "Paris Agreement", "North American Free Trade Agreement",
    "Maastricht Treaty", "Basel III",
]

wiki = wikipediaapi.Wikipedia(language="en", user_agent="NER-Dataset-Builder/1.0") # folosim alt user agent pt a evita blocarea de catre server (crede ca e un script)

wiki_texts = []

for title in tqdm(WIKI_TOPICS[:MAX_WIKI_ARTICLES], desc="Wikipedia"):
    try:
        page = wiki.page(title)     # extrage doar paginile wikipedia care au titlul ca acela din WIKI_TOPICS
        if page.exists() and len(page.text) > 500:
            wiki_texts.append({"title": title, "text": page.text[:5000]})  # primii 5000 chars
        time.sleep(0.2)  # politicos fata de serverele Wikipedia
    except Exception as e:
        print(f"Eroare la '{title}': {e}")

print(f"Colectate {len(wiki_texts)} articole Wikipedia.")
with open("dataset/raw/wikipedia.json", "w") as f:
    json.dump(wiki_texts, f)
print("Salvat: dataset/raw/wikipedia.json")

In [ ]:
# ─── Sursa 3: SEC EDGAR ───────────────────────────────────────────────────────
# SEC (Security and Exchange Commision) EDGAR contine rapoartele financiare ale companiilor listate la bursa.
# Acestea contin entitati economice dense: indicatori, organizatii, politici.
#
# NOTA: Daca sec-edgar-downloader ridica probleme in Colab,
# comenteaza aceasta celula — ai suficiente date din CC-News si Wikipedia.

try:
    from sec_edgar_downloader import Downloader
    import os

    # Cateva companii mari cu rapoarte publice relevante
    TICKERS = ["AAPL", "JPM", "GS", "BAC", "MS", "WFC", "C", "BLK"]

    dl = Downloader("NER Research", "ner@research.com", "./dataset/raw/sec_raw")
    sec_texts = []

    for ticker in tqdm(TICKERS[:5], desc="SEC EDGAR"):  # limitam la 5 pentru viteza
        try:
            dl.get("10-K", ticker, limit=1)  # annual report; 10-K codul oficial pt raportul financiar anual (toate companiile din SUA sunt obligate sa le depuna la SEC)
        except Exception as e:
            print(f"Skip {ticker}: {e}")

    # Citim fisierele descarcate si extragem text
    sec_dir = Path("./dataset/raw/sec_raw")
    if sec_dir.exists():
        for txt_file in list(sec_dir.rglob("*.txt"))[:MAX_SEC_FILINGS]: 
            try:
                text = txt_file.read_text(errors="ignore")[:3000]
                if len(text) > 200:
                    sec_texts.append(text)
            except:
                pass

    print(f"Colectate {len(sec_texts)} fragmente SEC EDGAR.")
    with open("dataset/raw/sec_edgar.json", "w") as f:
        json.dump(sec_texts, f)
    print("Salvat: dataset/raw/sec_edgar.json")

except Exception as e:
    print(f"SEC EDGAR nu e disponibil: {e}")
    print("Continuam fara SEC EDGAR — avem suficiente date din celelalte surse.")
    sec_texts = []

## PASUL 4 — Segmentare in propozitii

NER opereaza la nivel de propozitie/fraza, nu la nivel de document.
Segmentam toate documentele colectate si filtram propozitiile prea scurte sau prea lungi.

**Output:** un DataFrame `df_sentences` cu toate propozitiile relevante.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])  # doar sentence segmentation

def extract_sentences(texts, source_name, max_per_doc=MAX_SENTENCES_PER_DOC):
    """Extrage propozitii din lista de texte. Filtreaza dupa lungime."""
    sentences = []
    for text in tqdm(texts, desc=f"Segmentare {source_name}", leave=False):
        if isinstance(text, dict):
            text = text.get("text", "")
        try:
            doc = nlp(text[:10000])  # limita procesare per doc
            sents = [
                sent.text.strip()
                for sent in doc.sents
                if MIN_SENTENCE_LEN <= len(sent.text.strip()) <= MAX_SENTENCE_LEN
            ]
            sentences.extend(sents[:max_per_doc])
        except Exception:
            pass
    return sentences

# Incarcam datele raw (in caz ca ai restartat runtime-ul)
with open("dataset/raw/cc_news.json") as f:
    cc_news_texts = json.load(f)
with open("dataset/raw/wikipedia.json") as f:
    wiki_texts = json.load(f)

try:
    with open("dataset/raw/sec_edgar.json") as f:
        sec_texts = json.load(f)
except FileNotFoundError:
    sec_texts = []

# Segmentam fiecare sursa
sents_cc   = extract_sentences(cc_news_texts, "CC-News")
sents_wiki = extract_sentences(wiki_texts, "Wikipedia")
sents_sec  = extract_sentences(sec_texts, "SEC EDGAR")

# Combinam si deduplicam
all_sentences = list(set(sents_cc + sents_wiki + sents_sec))
random.shuffle(all_sentences)

# Cream DataFrame
df_sentences = pd.DataFrame({"text": all_sentences})
df_sentences = df_sentences[df_sentences["text"].str.len().between(MIN_SENTENCE_LEN, MAX_SENTENCE_LEN)]
df_sentences = df_sentences.drop_duplicates(subset="text").reset_index(drop=True)

print(f"Total propozitii dupa segmentare si deduplicare: {len(df_sentences)}")
print(f"  - CC-News:   {len(sents_cc)} propozitii")
print(f"  - Wikipedia: {len(sents_wiki)} propozitii")
print(f"  - SEC EDGAR: {len(sents_sec)} propozitii")
df_sentences.head(5)

## PASUL 5 — Weak Supervision cu Snorkel

Snorkel combina mai multe **Labeling Functions (LF)** — fiecare e o regula heuristica —
si produce pseudo-label-uri prin vot ponderat (Label Model).

**Tipuri de LF folosite:**
- **Regex**: pattern-uri pentru indicatori economici, legislatie, valute
- **Gazetteers**: liste de entitati cunoscute (politicieni, organizatii)
- **Wikidata**: interogare SPARQL pentru liste extinse de entitati (standard W3C limbaj de interogare destinat manipularii datelor din formatul Resource Description Framework RDF)
- **Heuristici contextuale**: "President X", "Minister Y"

**Important:** Snorkel lucreaza la nivel de propozitie (eticheteaza propozitia),
nu la nivel de token. In pasul urmator vom extrage span-urile exacte. 

Un span = secventa de cuvinte (tokeni) extrasa dintr-un text mare ce descrie perfect entitatea cautata.

In [ ]:
# ─── Gazetteers (liste de entitati cunoscute) ─────────────────────────────────
# Aceste liste servesc drept baza de cunostinte pentru LF-uri.
# Cu cat sunt mai complete, cu atat LF-urile sunt mai precise.

POLITICIANS_GAZETTEER = [
    "Joe Biden", "Donald Trump", "Barack Obama", "George W. Bush",
    "Bill Clinton", "Jerome Powell", "Janet Yellen", "Christine Lagarde",
    "Emmanuel Macron", "Olaf Scholz", "Rishi Sunak", "Angela Merkel",
    "Boris Johnson", "Xi Jinping", "Vladimir Putin", "Narendra Modi",
    "Ursula von der Leyen", "Mario Draghi", "Ben Bernanke", "Alan Greenspan",
    "Larry Summers", "Paul Volcker", "Mark Carney", "Lagarde",
    "Yellen", "Powell", "Bernanke",  # si forme scurte
]

FINANCIAL_ORGS_GAZETTEER = [
    "Federal Reserve", "Fed", "IMF", "International Monetary Fund",
    "World Bank", "European Central Bank", "ECB", "Bank of England",
    "Goldman Sachs", "JPMorgan", "JPMorgan Chase", "Morgan Stanley",
    "BlackRock", "Citigroup", "Bank of America", "Wells Fargo",
    "Deutsche Bank", "Barclays", "HSBC", "Credit Suisse",
    "Bank for International Settlements", "BIS", "FDIC",
    "Securities and Exchange Commission", "SEC",
]

POLITICAL_ORGS_GAZETTEER = [
    "NATO", "United Nations", "UN", "European Union", "EU",
    "European Commission", "European Parliament", "G7", "G20", "BRICS",
    "OECD", "WTO", "World Trade Organization", "Republican Party",
    "Democratic Party", "Labour Party", "Conservative Party",
    "Congress", "Senate", "House of Representatives", "Parliament",
    "White House", "Kremlin", "Bundestag",
]

ECONOMIC_INDICATORS_GAZETTEER = [
    "GDP", "GNP", "CPI", "PPI", "PCE", "PMI",
    "inflation rate", "inflation", "deflation", "stagflation",
    "unemployment rate", "unemployment", "jobless rate",
    "interest rate", "federal funds rate", "discount rate",
    "bond yield", "Treasury yield", "yield curve",
    "trade deficit", "trade surplus", "current account",
    "budget deficit", "national debt", "debt-to-GDP",
]

LEGISLATION_GAZETTEER = [
    "Dodd-Frank", "Dodd–Frank", "Glass-Steagall", "Basel III", "Basel IV",
    "Inflation Reduction Act", "CARES Act", "American Rescue Plan",
    "Sarbanes-Oxley", "Volcker Rule", "GDPR", "MiFID",
    "Patriot Act", "Affordable Care Act", "Stimulus Package",
]

CURRENCIES_GAZETTEER = [
    "USD", "EUR", "GBP", "JPY", "CNY", "CHF", "CAD", "AUD",
    "Bitcoin", "BTC", "Ethereum", "ETH", "dollar", "euro",
    "pound sterling", "yen", "yuan", "renminbi",
]

TRADE_AGREEMENTS_GAZETTEER = [
    "NAFTA", "USMCA", "TPP", "CPTPP", "RCEP",
    "Paris Agreement", "Maastricht Treaty", "Kyoto Protocol",
    "WTO Agreement", "GATT", "TTIP", "CETA",
]

MARKET_EVENTS_GAZETTEER = [
    "2008 financial crisis", "Great Recession", "dot-com bubble",
    "Black Monday", "Black Tuesday", "Great Depression",
    "European debt crisis", "Asian financial crisis",
    "Brexit", "COVID-19 recession", "subprime mortgage crisis",
]

print("Gazetteers incarcate.")
print(f"  Politicieni: {len(POLITICIANS_GAZETTEER)}")
print(f"  Org. financiare: {len(FINANCIAL_ORGS_GAZETTEER)}")
print(f"  Org. politice: {len(POLITICAL_ORGS_GAZETTEER)}")
print(f"  Indicatori econ.: {len(ECONOMIC_INDICATORS_GAZETTEER)}")

In [ ]:
# ─── Interogare Wikidata pentru politicieni suplimentari ─────────────────────
# Wikidata ne da o lista mult mai extinsa de politicieni decat gazetteer-ul manual.
# NOTA: Aceasta celula face o cerere HTTP catre Wikidata — poate dura 10-30s.

def get_wikidata_politicians(limit=3000):
    """Interogheaza Wikidata pentru lista de politicieni in engleza."""
    query = f"""
    SELECT DISTINCT ?personLabel WHERE {{
      ?person wdt:P31 wd:Q5 .
      ?person wdt:P106 wd:Q82955 .
      ?person wikibase:sitelinks ?links .
      FILTER(?links > 5)
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
    }} LIMIT {limit}
    """
    headers = {
        "User-Agent": "NER-Dataset-Builder/1.0 (research project)",
        "Accept": "application/json"
    }
    try:
        r = requests.get(
            "https://query.wikidata.org/sparql",
            params={"query": query, "format": "json"},
            headers=headers,
            timeout=30
        )
        if r.status_code == 200:
            names = [
                item["personLabel"]["value"]
                for item in r.json()["results"]["bindings"]
                if len(item["personLabel"]["value"].split()) >= 2  # doar nume compuse
            ]
            return names
    except Exception as e:
        print(f"Wikidata query esuat: {e}")
    return []

print("Se interogheaza Wikidata...")
wikidata_politicians = get_wikidata_politicians(limit=3000)
# Combinam cu gazetteer-ul manual
ALL_POLITICIANS = list(set(POLITICIANS_GAZETTEER + wikidata_politicians))
# Sortam dupa lungime descrescatoare pentru matching corect (evita match partial)
ALL_POLITICIANS.sort(key=len, reverse=True)

print(f"Total politicieni in baza de cunostinte: {len(ALL_POLITICIANS)}")

In [ ]:
# ─── Labeling Functions Snorkel ───────────────────────────────────────────────
# Fiecare LF returneaza un label (0-9) sau ABSTAIN (-1).
# ABSTAIN inseamna ca LF-ul nu e sigur — Snorkel ignora acest vot.
# Label Model combina voturile tuturor LF-urilor.

from snorkel.labeling import labeling_function
from snorkel.labeling import PandasLFApplier
from snorkel.labeling.model import LabelModel

ABSTAIN          = -1
L_POLITICIAN     = 0
L_POLITICAL_PARTY= 1
L_POLITICAL_ORG  = 2
L_FINANCIAL_ORG  = 3
L_ECONOMIC_IND   = 4
L_POLICY         = 5
L_LEGISLATION    = 6
L_MARKET_EVENT   = 7
L_CURRENCY       = 8
L_TRADE_AGREEMENT= 9

CARDINALITY = 10  # numarul de labeluri (fara ABSTAIN)

# --- LF 1: Politicieni din gazetteer + Wikidata ---
@labeling_function()
def lf_politician_gazetteer(x):
    for name in ALL_POLITICIANS:
        if name.lower() in x.text.lower():
            return L_POLITICIAN
    return ABSTAIN

# --- LF 2: Heuristica titlu + nume pentru politicieni ---
@labeling_function()
def lf_politician_title(x): 
    # \b -> extrage exact cuvintele respective (cu tot cu spatiile inserate)
    # \s -> spatiu (+ = unul sau mai multe = cel putin unul)
    #  apoi o litera sau mai multe
    pattern = r"\b(President|Vice President|Senator|Prime Minister|Chancellor|" \
              r"Secretary of State|Foreign Minister|Finance Minister|Governor|" \
              r"Congressman|Congresswoman|Representative)\s+[A-Z][a-z]+"
    return L_POLITICIAN if re.search(pattern, x.text) else ABSTAIN

# --- LF 3: Partide politice ---
@labeling_function()
def lf_political_party(x):
    parties = ["Republican Party", "Democratic Party", "Labour Party",
               "Conservative Party", "Liberal Party", "Green Party",
               "Socialist Party", "Democrats", "Republicans"]
    return L_POLITICAL_PARTY if any(p.lower() in x.text.lower() for p in parties) else ABSTAIN

# --- LF 4: Organizatii politice din gazetteer ---
@labeling_function()
def lf_political_org_gazetteer(x):
    return L_POLITICAL_ORG if any(o.lower() in x.text.lower() for o in POLITICAL_ORGS_GAZETTEER) else ABSTAIN

# --- LF 5: Organizatii financiare din gazetteer ---
@labeling_function()
def lf_financial_org_gazetteer(x):
    return L_FINANCIAL_ORG if any(o.lower() in x.text.lower() for o in FINANCIAL_ORGS_GAZETTEER) else ABSTAIN

# --- LF 6: Regex pentru organizatii financiare ---
@labeling_function()
def lf_financial_org_regex(x):
    # s? la finalul expresiilor din regex cauta la final litera s sau nu. 
    # \b de la final cauta exact acea combinatie de pana la s?. (nu si expresii de tipul hedge funding)
    pattern = r"\b(central bank|investment bank|hedge fund|private equity|asset manager)s?\b"
    return L_FINANCIAL_ORG if re.search(pattern, x.text, re.IGNORECASE) else ABSTAIN

# --- LF 7: Indicatori economici din gazetteer ---
@labeling_function()
def lf_economic_indicator_gazetteer(x):
    return L_ECONOMIC_IND if any(i.lower() in x.text.lower() for i in ECONOMIC_INDICATORS_GAZETTEER) else ABSTAIN

# --- LF 8: Regex pentru indicatori economici cu valori numerice ---
@labeling_function()
def lf_economic_indicator_numeric(x):
    # Pattern: "GDP grew by 3.2%", "inflation hit 8%", "rate of 5.25%"
    # \b exact acele expresii/cuvinte
    # .{0,30} -> orice caractere ce pot aparea de 0 sau 30 de ori.
    # \d+ -> digit cel putin una
    # \.? -> un caracter punct sau nu
    # \d* -> digit de 0 sau mai multe ori
    # \s* -> spatiu de 0 sau mai multe ori
    # % -> caracter efectiv
    pattern = r"\b(GDP|inflation|unemployment|growth|rate|yield)\b.{0,30}\d+\.?\d*\s*%"
    return L_ECONOMIC_IND if re.search(pattern, x.text, re.IGNORECASE) else ABSTAIN

# --- LF 9: Politici monetare/fiscale ---
@labeling_function()
def lf_policy(x):
    policies = ["quantitative easing", "quantitative tightening", "green new deal",
                "austerity", "stimulus", "bailout", "tapering", "rate hike", "rate cut",
                "fiscal policy", "monetary policy", "economic policy",
                "budget", "tax reform", "deficit spending"]
    return L_POLICY if any(p.lower() in x.text.lower() for p in policies) else ABSTAIN

# --- LF 10: Legislatie ---
@labeling_function()
def lf_legislation(x):
    # Pattern general: "X Act", "X Law", "X Bill"
    # cauta orice tip de text ce contine litere A-Za-z sau spatiu sau - ce apare cel putin o data
    # apoi cauta exact acele cuvinte
    act_pattern = r"\b[A-Z][a-zA-Z\s-]+\b(Act|Law|Bill|Regulation|Directive|Amendment)\b"
    if re.search(act_pattern, x.text):
        return L_LEGISLATION
    return L_LEGISLATION if any(l.lower() in x.text.lower() for l in LEGISLATION_GAZETTEER) else ABSTAIN

# --- LF 11: Evenimente de piata ---
@labeling_function()
def lf_market_event(x):
    return L_MARKET_EVENT if any(e.lower() in x.text.lower() for e in MARKET_EVENTS_GAZETTEER) else ABSTAIN

# --- LF 12: Regex pentru crize financiare ---
@labeling_function()
def lf_market_event_regex(x):
    # matching exact pe acele expresii formate din mai multe cuvinte cu spatii intre ele.
    pattern = r"\b(financial crisis|market crash|stock market|bear market|bull market|recession|depression)\b"
    return L_MARKET_EVENT if re.search(pattern, x.text, re.IGNORECASE) else ABSTAIN

# --- LF 13: Valute ---
@labeling_function()
def lf_currency(x):
    if any(c in x.text for c in CURRENCIES_GAZETTEER):
        return L_CURRENCY
    # Pattern: "$1.2 trillion", "€500 billion", "¥100"
    # unul din acele caractere valuta apoi 0 sau mai multe spatii, apoi cel putin o cifra
    pattern = r"[\$€£¥]\s*\d+"
    return L_CURRENCY if re.search(pattern, x.text) else ABSTAIN

# --- LF 14: Acorduri comerciale ---
@labeling_function()
def lf_trade_agreement(x):
    return L_TRADE_AGREEMENT if any(t.lower() in x.text.lower() for t in TRADE_AGREEMENTS_GAZETTEER) else ABSTAIN

ALL_LFS = [
    lf_politician_gazetteer, lf_politician_title, lf_political_party,
    lf_political_org_gazetteer, lf_financial_org_gazetteer, lf_financial_org_regex,
    lf_economic_indicator_gazetteer, lf_economic_indicator_numeric,
    lf_policy, lf_legislation, lf_market_event, lf_market_event_regex,
    lf_currency, lf_trade_agreement
]

print(f"Definite {len(ALL_LFS)} Labeling Functions.")

In [ ]:
# ─── Aplicare LF-uri si Label Model ──────────────────────────────────────────
# PandasLFApplier aplica toate LF-urile pe fiecare rand din DataFrame.
# Rezultatul L_matrix are shape (n_sentences, n_lfs) cu voturi per LF.
# LabelModel combina aceste voturi si produce pseudo-label-uri finale.

print("Se aplica Labeling Functions...")
applier = PandasLFApplier(lfs=ALL_LFS)
L_matrix = applier.apply(df=df_sentences)

print(f"Shape L_matrix: {L_matrix.shape}")
print(f"  Rows: {L_matrix.shape[0]} propozitii")
print(f"  Cols: {L_matrix.shape[1]} LF-uri")

# Statistici LF-uri — coverage si conflicts
from snorkel.labeling import LFAnalysis
lf_analysis = LFAnalysis(L=L_matrix, lfs=ALL_LFS).lf_summary()
print("\nAnaliza LF-uri:")
print(lf_analysis.to_string())

In [ ]:
# ─── Antrenare Label Model ────────────────────────────────────────────────────
# LabelModel invata ponderi pentru fiecare LF pe baza corelatiilor dintre ele.
# LF-urile mai precise primesc ponderi mai mari.

print("Se antreneaza Label Model...")
label_model = LabelModel(cardinality=CARDINALITY, verbose=True) # nu e un model de regresie 
            # model generativ din Snorkel ce observa matricea de voturi L_matrix si invata din felul in care s-au aplicat
            # Label Functions asupra textului.
            # Dacă LF1 și LF2 dau adesea aceeași etichetă, modelul presupune că ambele sunt corecte.
            # Dacă LF3 dă o etichetă, dar LF1 și LF2 zic că se abțin, modelul învață să aibă încredere în LF3 pentru acel caz specific.
            # Dacă LF4 greșește frecvent (intră în conflict cu celelalte), modelul îi va scădea ponderea (importanța). 

# modelul se uita doar la matricea de voturi pt a estima cat de buna este fiecare regula. 
label_model.fit(
    L_train=L_matrix,
    n_epochs=500,
    lr=0.001,
    log_freq=100,
    seed=42
)

# Predictii cu probabilitati
pseudo_labels      = label_model.predict(L=L_matrix)
pseudo_probs       = label_model.predict_proba(L=L_matrix)

# aici se adauga la dataframe 2 coloane: snorkel_label adica eticheta catre care textul apartine de entitatea X.
# snorkel_confidence adica probabilitatea de apartenenta la entitatea X.  
df_sentences["snorkel_label"]      = pseudo_labels
df_sentences["snorkel_confidence"] = pseudo_probs.max(axis=1)

# Filtram: pastram doar propozitiile cu label non-ABSTAIN si confidence > 0.7   
# la noi ABSTAIN ar putea fi considerata o clasa NAN
df_weak = df_sentences[
    (df_sentences["snorkel_label"] != -1) &
    (df_sentences["snorkel_confidence"] >= 0.7)
].copy()

# Mapam label index → label string ca sa avem etichete String (sa putem vedea si in dataframe).
IDX_TO_LABEL = {v: k for k, v in {
    "POLITICIAN": 0, "POLITICAL_PARTY": 1, "POLITICAL_ORG": 2,
    "FINANCIAL_ORG": 3, "ECONOMIC_INDICATOR": 4, "POLICY": 5,
    "LEGISLATION": 6, "MARKET_EVENT": 7, "CURRENCY": 8, "TRADE_AGREEMENT": 9
}.items()}

df_weak["label"] = df_weak["snorkel_label"].map(IDX_TO_LABEL)

print(f"\nPropozitii cu pseudo-label dupa filtrare: {len(df_weak)}")
print("\nDistributie labeluri:")
print(df_weak["label"].value_counts().to_string())

## PASUL 6 — Extractie entitati (Span Detection)

Snorkel ne-a spus *ce tip* de entitate contine o propozitie.
Acum trebuie sa identificam *unde exact* se afla entitatea in text (start, end char).

Folosim gazetteer matching + regex pentru a extrage span-uri precise.
Output-ul va fi in format JSON compatibil cu GLiNER si Doccano.

In [ ]:
# ─── Extractie span-uri exacte din propozitii ────────────────────────────────

LABEL_TO_GAZETTEER = {
    "POLITICIAN":        ALL_POLITICIANS,
    "POLITICAL_ORG":     POLITICAL_ORGS_GAZETTEER,
    "FINANCIAL_ORG":     FINANCIAL_ORGS_GAZETTEER,
    "ECONOMIC_INDICATOR": ECONOMIC_INDICATORS_GAZETTEER,
    "LEGISLATION":       LEGISLATION_GAZETTEER,
    "CURRENCY":          CURRENCIES_GAZETTEER,
    "TRADE_AGREEMENT":   TRADE_AGREEMENTS_GAZETTEER,
    "MARKET_EVENT":      MARKET_EVENTS_GAZETTEER,
}

def find_entity_spans(text, label):
    """Gaseste toate aparitiile entitatii in text si returneaza span-uri."""
    spans = []
    gazetteer = LABEL_TO_GAZETTEER.get(label, [])

    for entity in gazetteer:
        # Case-insensitive match cu word boundaries
        # cauta exact cuvantul din mijloc, iar escape converteste acel cuvant in string pt a evita cazurile in care contine ,.- etc. 

        pattern = r"\b" + re.escape(entity) + r"\b"
        for match in re.finditer(pattern, text, re.IGNORECASE): # ignora diferentele intre literele mari si mici.
            spans.append({
                "text":  match.group(),
                "label": label,
                "start": match.start(),
                "end":   match.end()
            })

    # Policy: pattern special (nu e in gazetteer fix)
    if label == "POLICY":
        policy_patterns = [
            r"\b(quantitative easing|quantitative tightening|rate hike|rate cut|" \
            r"fiscal stimulus|austerity measures|tapering|bailout)\b"
        ]
        for pat in policy_patterns:
            for match in re.finditer(pat, text, re.IGNORECASE):
                spans.append({"text": match.group(), "label": "POLICY",
                               "start": match.start(), "end": match.end()})

    # Elimina duplicate si overlap-uri
    spans.sort(key=lambda s: (s["start"], -(s["end"]-s["start"])))
    unique = []
    last_end = -1
    for s in spans:
        if s["start"] >= last_end:
            unique.append(s)
            last_end = s["end"]
    return unique


def convert_to_ner_format(df):
    """Converteste DataFrame Snorkel in lista de exemple NER cu span-uri."""
    ner_examples = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extractie entitati"):
        spans = find_entity_spans(row["text"], row["label"])
        if spans:  # pastram doar exemplele cu cel putin o entitate identificata
            ner_examples.append({
                "text":     row["text"],
                "entities": spans,
                "source":   "snorkel_weak_supervision"
            })
    return ner_examples


ner_weak = convert_to_ner_format(df_weak)

# Salvam
with open("dataset/annotated/weak_supervision.jsonl", "w") as f:
    for ex in ner_weak:
        f.write(json.dumps(ex) + "\n")

print(f"Exemple NER cu span-uri extrase: {len(ner_weak)}")
print("\nPrime 2 exemple:")
for ex in ner_weak[:2]:
    print(json.dumps(ex, indent=2))
    print()

## PASUL 7 — Adnotare LLM cu Claude API (~2000 exemple)  ELIMINAT

Folosim Claude pentru a genera adnotari NER de calitate mai buna decat weak supervision.

### Cum obtii API Key-ul:
1. Mergi la **https://console.anthropic.com**
2. Creezi un cont sau te loghezi
3. Din meniul din stanga: **API Keys** > **Create Key**
4. Copiezi cheia (incepe cu `sk-ant-...`)
5. **In Colab:** mergi la pictograma 🔑 (Secrets) din bara stanga > adaugi secret cu numele `ANTHROPIC_API_KEY`
6. **Local:** setezi variabila de mediu: `export ANTHROPIC_API_KEY=sk-ant-...`

⚠️ **Nu pune niciodata API key-ul direct in cod!** Foloseste Secrets (Colab) sau variabile de mediu.

In [ ]:
# ─── Configurare API Key ──────────────────────────────────────────────────────
import os

# Incercam sa citim din Colab Secrets, apoi din variabila de mediu
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
    print("API Key citit din Colab Secrets.")
except Exception:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    if ANTHROPIC_API_KEY:
        print("API Key citit din variabila de mediu.")
    else:
        print("ATENTIE: API Key nu a fost gasit.")
        print("Adauga-l in Colab Secrets (pictograma cheie din bara stanga)")
        print("cu numele ANTHROPIC_API_KEY sau seteaza variabila de mediu.")

if ANTHROPIC_API_KEY:
    print(f"API Key detectat: sk-ant-...{ANTHROPIC_API_KEY[-6:]}")
else:
    raise ValueError("API Key lipsa. Urmeaza instructiunile de mai sus.")

In [ ]:
# ─── Adnotare NER cu Claude ───────────────────────────────────────────────────
# Trimitem propozitii neadnotate catre Claude si cerem adnotari NER in JSON.
# Folosim batch-uri de 5 propozitii per request pentru eficienta si cost redus.
#
# Cost estimat: ~2000 propozitii * ~200 tokens/request = ~400K tokens
# La pretul Claude Haiku: ~$0.04 (foarte ieftin)
# Daca vrei sa reduci costul, scade LLM_ANNOTATION_TARGET.

import anthropic
import time

LLM_ANNOTATION_TARGET = 2000   # numarul de exemple dorit
BATCH_SIZE            = 5      # propozitii per request API
RETRY_DELAY           = 2      # secunde intre retries

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """You are a precise NER annotation expert. 
You annotate text with these entity labels ONLY:
POLITICIAN, POLITICAL_PARTY, POLITICAL_ORG, FINANCIAL_ORG,
ECONOMIC_INDICATOR, POLICY, LEGISLATION, MARKET_EVENT, CURRENCY, TRADE_AGREEMENT, GPE.

Rules:
- Annotate ONLY entities that clearly belong to the listed categories
- Use character-level start/end offsets (0-indexed)
- A sentence may have 0 or multiple entities
- GPE = geopolitical entity (country, region, bloc like "EU", "Eurozone")
- Return ONLY valid JSON, no explanation, no markdown"""

USER_PROMPT_TEMPLATE = """Annotate the following sentences. Return a JSON array where each element has:
{{"text": "original sentence", "entities": [{{"text": "entity text", "label": "LABEL", "start": 0, "end": 5}}]}}

Sentences:
{sentences}"""


def annotate_batch(sentences: list[str]) -> list[dict]:
    """Trimite un batch de propozitii la Claude si returneaza adnotarile."""
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
    prompt   = USER_PROMPT_TEMPLATE.format(sentences=numbered)

    for attempt in range(3):  # 3 retries
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",  # cel mai rapid si ieftin
                max_tokens=2000,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}]
            )
            raw = response.content[0].text.strip()
            # Curatam eventuale markdown code blocks
            raw = re.sub(r"```json|```", "", raw).strip()
            parsed = json.loads(raw)
            # Validam si adaugam sursa
            results = []
            for item in parsed:
                if "text" in item and "entities" in item:
                    item["source"] = "llm_claude"
                    results.append(item)
            return results
        except json.JSONDecodeError:
            if attempt < 2:
                time.sleep(RETRY_DELAY)
        except anthropic.RateLimitError:
            time.sleep(10)
        except Exception as e:
            if attempt < 2:
                time.sleep(RETRY_DELAY)
    return []  # returnam lista goala daca toate retry-urile esueaza


# ─── Selectam propozitii diverse pentru adnotare LLM ─────────────────────────
# Folosim propozitii din corpus care NU au fost adnotate de Snorkel (sau cu conf. mica)
# pentru a maximiza diversitatea dataset-ului.

annotated_texts = set(ex["text"] for ex in ner_weak)
candidates_llm  = df_sentences[
    ~df_sentences["text"].isin(annotated_texts)
]["text"].tolist()

random.shuffle(candidates_llm)
candidates_llm = candidates_llm[:LLM_ANNOTATION_TARGET * 2]  # buffer

print(f"Propozitii candidate pentru adnotare LLM: {len(candidates_llm)}")
print(f"Target: {LLM_ANNOTATION_TARGET} exemple adnotate")
print(f"Batch size: {BATCH_SIZE} | Batches estimate: {LLM_ANNOTATION_TARGET // BATCH_SIZE}")

In [ ]:
# ─── Rulare adnotare LLM ─────────────────────────────────────────────────────
# Aceasta celula face ~400 request-uri API. Dureaza ~10-20 minute.
# Progresul e salvat continuu — daca se intrerupe, poti relua de unde ai ramas.

output_path = Path("dataset/annotated/llm_annotated.jsonl")

# Verificam daca exista adnotari anterioare (resume)
existing_llm = []
if output_path.exists():
    with open(output_path) as f:
        existing_llm = [json.loads(l) for l in f if l.strip()]
    print(f"Gasit {len(existing_llm)} adnotari anterioare. Se continua...")

existing_texts = set(ex["text"] for ex in existing_llm)
remaining      = [s for s in candidates_llm if s not in existing_texts]
llm_examples   = existing_llm.copy()

with open(output_path, "a") as f_out:
    batches = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]

    pbar = tqdm(batches, desc="LLM Annotation")
    for batch in pbar:
        if len(llm_examples) >= LLM_ANNOTATION_TARGET:
            break

        results = annotate_batch(batch)

        for ex in results:
            if ex.get("entities"):  # pastram doar exemplele cu cel putin o entitate
                f_out.write(json.dumps(ex) + "\n")
                llm_examples.append(ex)

        pbar.set_postfix({"total": len(llm_examples), "target": LLM_ANNOTATION_TARGET})
        time.sleep(0.3)  # rate limiting politicos

print(f"\nAdnotare LLM completa: {len(llm_examples)} exemple")
print("Salvat: dataset/annotated/llm_annotated.jsonl")

# Preview
print("\nPrime 2 exemple adnotate de Claude:")
for ex in llm_examples[:2]:
    print(json.dumps(ex, indent=2))
    print()

## PASUL 8 — Adnotare manuala cu Doccano (Gold Standard)        ELIMINAT

Adnotarea manuala produce cel mai **precis** subset al dataset-ului (~500-1000 exemple).
Acesta va fi folosit drept **test set** (gold standard) pentru evaluarea corecta a modelului.

### Setup Doccano in Google Colab:

**NOTA:** Doccano necesita sa fie accesat printr-un browser. In Colab folosim `ngrok` pentru
a expune serverul local la un URL public temporar.

**Pasi:**
1. Ruleaza celula de mai jos
2. Copiaza URL-ul ngrok afisat
3. Deschide URL-ul in browser
4. Login: `admin` / `password` (schimba-le!)
5. Importa fisierul `doccano_import.jsonl` generat
6. Adnoteaza 500-1000 propozitii
7. Exporta in format JSONL si salveaza ca `dataset/annotated/gold_standard.jsonl`

**Alternativa locala (recomandata):**
```bash
pip install doccano
doccano init
doccano createuser --username admin --password admin
doccano webserver --port 8000
# Acceseaza http://localhost:8000
```

In [ ]:
# ─── Pregatire fisier import Doccano ─────────────────────────────────────────
# Selectam propozitii reprezentative pentru adnotare manuala.
# Alegem diversitate: propozitii din surse diferite, cu entitati diferite.

# Selectam propozitii reprezentative — unele cu entitati probabile, altele random
high_value = df_sentences[
    df_sentences["text"].str.contains(
        "|".join(["Fed", "GDP", "president", "election", "IMF", "Congress",
                  "inflation", "policy", "sanctions", "treaty"]),
        case=False, regex=True
    )
]["text"].tolist()

random_sample = df_sentences["text"].sample(n=min(200, len(df_sentences)), random_state=42).tolist()
doccano_candidates = list(set(high_value[:800] + random_sample))
random.shuffle(doccano_candidates)
doccano_candidates = doccano_candidates[:1000]  # maxim 1000 pentru adnotare

# Format Doccano: JSONL cu campul 'text'
doccano_path = Path("dataset/annotated/doccano_import.jsonl")
with open(doccano_path, "w") as f:
    for text in doccano_candidates:
        f.write(json.dumps({"text": text}) + "\n")

print(f"Generat fisier pentru import Doccano: {doccano_path}")
print(f"Propozitii de adnotat: {len(doccano_candidates)}")
print(f"\nLabeluri de folosit in Doccano (exact cum sunt definite):")
for label in NER_LABELS:
    print(f"  - {label}")

In [ ]:
# ─── Pornire Doccano in Colab cu ngrok ───────────────────────────────────────
# NOTA: Necesita un cont ngrok gratuit. Inregistreaza-te la https://ngrok.com
# si obtine authtoken-ul din dashboard.
#
# Daca preferi sa adnotezi local (recomandat), sari aceasta celula
# si foloseste instructiunile din celula de markdown de mai sus.

NGROK_TOKEN = "3CWga7JtrRKSyN7gjIoCo9B7en1_7FJ5CryF5GNo6rwuAGdYE"  # Pune aici token-ul tau ngrok sau adauga-l in Colab Secrets

try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_TOKEN", NGROK_TOKEN)
except:
    pass

if NGROK_TOKEN:
    !pip install -q doccano pyngrok
    !doccano init
    !doccano createuser --username admin --password admin123

    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN

    import subprocess, threading
    def run_doccano():
        subprocess.run(["doccano", "webserver", "--port", "8000"])

    t = threading.Thread(target=run_doccano, daemon=True)
    t.start()
    time.sleep(5)

    tunnel = ngrok.connect(8000)
    print(f"\nDoccano disponibil la: {tunnel.public_url}")
    print("Username: admin | Password: admin123")
    print(f"Importa fisierul: {doccano_path.resolve()}")
else:
    print("NGROK_TOKEN nu e setat.")
    print("Optiunea recomandata: adnoteaza local cu:")
    print("  pip install doccano")
    print("  doccano init")
    print("  doccano createuser --username admin --password admin")
    print("  doccano webserver --port 8000")
    print(f"  # Importa: {doccano_path}")

In [ ]:
# ─── [DUPA ADNOTARE] Incarcare Gold Standard ──────────────────────────────────
# Ruleaza aceasta celula DUPA ce ai exportat din Doccano si ai salvat fisierul.
#
# Format export Doccano NER: JSONL cu campurile 'text' si 'label'
# Doccano exporta in format: {"text": "...", "label": [[start, end, "LABEL"], ...]}
#
# Daca nu ai terminat adnotarea, celula va continua cu un gold standard gol
# si va afisa un avertisment.

gold_path = Path("dataset/annotated/gold_standard.jsonl")

def convert_doccano_export(doccano_jsonl_path, output_path):
    """Converteste exportul Doccano in formatul nostru NER standard."""
    converted = []
    with open(doccano_jsonl_path) as f:
        for line in f:
            item = json.loads(line.strip())
            entities = []
            for ann in item.get("label", []):
                start, end, label = ann[0], ann[1], ann[2]
                entities.append({
                    "text":  item["text"][start:end],
                    "label": label,
                    "start": start,
                    "end":   end
                })
            converted.append({
                "text":     item["text"],
                "entities": entities,
                "source":   "manual_gold_standard"
            })
    with open(output_path, "w") as f:
        for ex in converted:
            f.write(json.dumps(ex) + "\n")
    return converted

# Cauta exportul Doccano (poate fi salvat cu diferite nume)
doccano_export = None
for candidate in ["dataset/annotated/doccano_export.jsonl",
                  "dataset/annotated/gold_standard_raw.jsonl",
                  "doccano_export.jsonl"]:
    if Path(candidate).exists():
        doccano_export = candidate
        break

if doccano_export:
    gold_examples = convert_doccano_export(doccano_export, gold_path)
    print(f"Gold standard incarcat: {len(gold_examples)} exemple")
else:
    print("ATENTIE: Gold standard (Doccano export) nu a fost gasit.")
    print("Continuam fara gold standard — il poti adauga mai tarziu.")
    print("Dupa adnotare, salveaza exportul ca: dataset/annotated/doccano_export.jsonl")
    gold_examples = []

## PASUL 9 — Combinare si validare dataset final

Combinam cele trei surse de adnotari:
- **Weak supervision** (Snorkel): volum mare, calitate medie
- **LLM annotated** (Claude): volum mediu, calitate buna
- **Gold standard** (Doccano): volum mic, calitate maxima

Aplicam filtre de calitate inainte de split.

In [ ]:
# ─── Incarcare toate sursele ─────────────────────────────────────────────────

def load_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

weak_examples  = load_jsonl("dataset/annotated/weak_supervision.jsonl")
llm_examples   = load_jsonl("dataset/annotated/llm_annotated.jsonl")
gold_examples  = load_jsonl("dataset/annotated/gold_standard.jsonl")

print("=== Statistici dataset ===\n")
print(f"Weak supervision (Snorkel): {len(weak_examples):>6} exemple")
print(f"LLM annotated (Claude):     {len(llm_examples):>6} exemple")
print(f"Gold standard (Doccano):    {len(gold_examples):>6} exemple")
print(f"{'─'*40}")
print(f"TOTAL:                      {len(weak_examples)+len(llm_examples)+len(gold_examples):>6} exemple")

In [ ]:
# ─── Validare si curatare ─────────────────────────────────────────────────────
# Aplicam filtre pentru a asigura calitatea dataset-ului.

def validate_example(ex):
    """Returneaza True daca exemplul e valid, False daca trebuie eliminat."""
    text = ex.get("text", "")
    if not text or len(text) < MIN_SENTENCE_LEN:    # eliminam daca are o lungime mai mica 
        return False

    entities = ex.get("entities", [])       # extrage doar atributul entities
    if not entities:  # fara entitati nu e util pentru NER
        return False

    for ent in entities:
        # Verificam ca span-urile sunt valide
        start, end = ent.get("start", 0), ent.get("end", 0)
        if start < 0 or end > len(text) or start >= end:
            return False
        # Verificam ca textul entitati corespunde pozitiei
        if ent.get("text") and ent["text"] != text[start:end]:
            return False
        # Verificam ca labelul e cunoscut
        if ent.get("label") and ent["label"] not in NER_LABELS + ["GPE"]:
            return False

    return True


def deduplicate(examples):
    """Elimina exemplele cu text duplicat."""
    seen, unique = set(), []
    for ex in examples:
        if ex["text"] not in seen:
            seen.add(ex["text"])
            unique.append(ex)
    return unique


# Combinam in ordinea: gold > llm > weak (prioritate la calitate mai mare)
all_examples = gold_examples + llm_examples + weak_examples

# Validare
valid_examples = [ex for ex in all_examples if validate_example(ex)]
print(f"Dupa validare: {len(valid_examples)} / {len(all_examples)} exemple")

# Deduplicare (gold standard are prioritate — apare primul)
unique_examples = deduplicate(valid_examples)
print(f"Dupa deduplicare: {len(unique_examples)} exemple")

# Shuffle final
random.shuffle(unique_examples)

# Distributie labeluri finale
from collections import Counter
label_counts = Counter(
    ent["label"]
    for ex in unique_examples
    for ent in ex["entities"]
)
print("\nDistributie entitati in dataset final:")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:<25} {count:>6} entitati")

## PASUL 10 — Train / Dev / Test Split

Strategia de split:
1. **Split initial**: 80% train_full + 20% test
2. **Split secundar pe train_full**: 80% train + 20% dev

**Rezultat final:**
- `train.jsonl`: 64% din total — pentru antrenare
- `dev.jsonl`:   16% din total — pentru validare in timpul antrenarii (early stopping)
- `test.jsonl`:  20% din total — pentru evaluarea finala a modelului (nu se atinge pana la final!)

**Regula importanta:** Gold standard (Doccano) merge **obligatoriu** in test set.

In [ ]:
# ─── Train / Dev / Test Split ────────────────────────────────────────────────

from sklearn.model_selection import train_test_split

# Gold standard merge direct in test — nu va fi vazut la antrenare
gold_texts = set(ex["text"] for ex in gold_examples)
test_gold  = [ex for ex in unique_examples if ex["text"] in gold_texts]
rest       = [ex for ex in unique_examples if ex["text"] not in gold_texts]

# ── Split 1: rest → 80% train_full + 20% test_extra ──────────────────────────
train_full, test_extra = train_test_split(
    rest,
    test_size=0.20,
    random_state=42
)

# Test final = gold standard + test extra
test_set = test_gold + test_extra
random.shuffle(test_set)

# ── Split 2: train_full → 80% train + 20% dev ────────────────────────────────
train_set, dev_set = train_test_split(
    train_full,
    test_size=0.20,
    random_state=42
)

# ── Salvare ──────────────────────────────────────────────────────────────────
def save_jsonl(examples, path):
    with open(path, "w") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")
    print(f"Salvat: {path} ({len(examples)} exemple)")

save_jsonl(train_set, "dataset/splits/train.jsonl")
save_jsonl(dev_set,   "dataset/splits/dev.jsonl")
save_jsonl(test_set,  "dataset/splits/test.jsonl")

# ── Sumar final ───────────────────────────────────────────────────────────────
total = len(train_set) + len(dev_set) + len(test_set)
print(f"""
=== SPLIT FINAL ===

  train.jsonl : {len(train_set):>5} exemple  ({len(train_set)/total*100:.1f}%)
  dev.jsonl   : {len(dev_set):>5} exemple  ({len(dev_set)/total*100:.1f}%)
  test.jsonl  : {len(test_set):>5} exemple  ({len(test_set)/total*100:.1f}%)
  ─────────────────────────────
  TOTAL       : {total:>5} exemple

  Din test set: {len(test_gold)} exemple gold standard (Doccano)
""")

In [ ]:
# ─── Statistici finale si verificare sanity ──────────────────────────────────
# Verificam ca split-urile nu au overlap si distributia labelurilor e echilibrata.

train_texts = set(ex["text"] for ex in train_set)
dev_texts   = set(ex["text"] for ex in dev_set)
test_texts  = set(ex["text"] for ex in test_set)

print("=== Verificare overlap (trebuie sa fie 0) ===")
print(f"  train ∩ dev  : {len(train_texts & dev_texts)}")
print(f"  train ∩ test : {len(train_texts & test_texts)}")
print(f"  dev   ∩ test : {len(dev_texts & test_texts)}")

print("\n=== Distributie entitati per split ===")
for split_name, split_data in [("train", train_set), ("dev", dev_set), ("test", test_set)]:
    counts = Counter(ent["label"] for ex in split_data for ent in ex["entities"])
    total_ents = sum(counts.values())
    print(f"\n  [{split_name}] {total_ents} entitati totale:")
    for label in NER_LABELS + ["GPE"]:
        if label in counts:
            print(f"    {label:<25} {counts[label]:>5}")

print("\n=== Structura directoare finale ===")
for p in sorted(Path("dataset").rglob("*.jsonl")):
    size = sum(1 for _ in open(p))
    print(f"  {p}  ({size} linii)")

print("\nDataset gata. Urmatorul pas: Fine-tuning GLiNER / spaCy.")